In [1]:
from analysis.processors.base import BaseProcessor
from coffea.nanoevents import NanoAODSchema
from coffea import processor
import uproot
import warnings
from coffea.dataset_tools import preprocess
warnings.filterwarnings(
    "ignore",
    category=RuntimeWarning,
    module="coffea.nanoevents.schemas.nanoaod",
)
NanoAODSchema.warn_missing_crossrefs = False
!voms-proxy-init --voms cms > /dev/null 2>&1

In [2]:
fileset = {
    '2017': {
        'SingleMuonB': {'SingleMuonB_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/data/Run2017B/SingleMuon/NANOAOD/UL2017_MiniAODv2_NanoAODv9-v1/280000/E2510F22-A4C8-214D-988D-E700218D843E.root': 'Events'},
            'metadata': {'short_name': 'SingleMuon'}}},
        'TTToHadronic': {'TTToHadronic_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/RunIISummer20UL17NanoAODv9/TTToHadronic_TuneCP5_13TeV-powheg-pythia8/NANOAODSIM/106X_mc2017_realistic_v9-v1/80000/D5FBA8C2-254A-3549-B013-3337F47575CB.root': 'Events'},
            'metadata': {'short_name': 'tt'}}},
    },
    '2022preEE': {
        'TTTo2L2Nu': {'TTTo2L2Nu_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/Run3Summer22NanoAODv12/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/50000/d51aa7d0-59ab-4ce7-8852-3f9a1ec14bc6.root': 'Events'},
            'metadata': {'short_name': 'tt'}}},
        'SingleMuonC': {'SingleMuonRun2022C_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/data/Run2022C/DoubleMuon/NANOAOD/22Sep2023-v1/50000/9f317280-5380-49f7-8a30-3e3bcd48dc9c.root': 'Events'},
            'metadata': {'short_name': 'SingleMuon'}}},
        'WJetsToLNu': {'WJetsToLNu_0to120_HT100to400_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/Run3Summer22NanoAODv12/WtoLNu-4Jets_MLNu-120_HT-1500to2500_TuneCP5_13p6TeV_madgraphMLM-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_v5-v2/40000/a008699b-a67b-4f2e-8712-7133198d0770.root': 'Events'},
            'metadata': {'short_name': 'WJetsToLNu'}}},
    },
    '2022postEE': {
        'TTtoLNu2Q': {'TTtoLNu2Q_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/Run3Summer22EENanoAODv12/TTtoLNu2Q_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/130X_mcRun3_2022_realistic_postEE_v6-v2/50000/3e4056b0-3ff6-44e2-b1e5-cf6db33d3a89.root': 'Events'},
            'metadata': {'short_name': 'tt'}}},
    },
    '2023postBPix': {
        'SingleElectronD': {'EGamma1v2D_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/data/Run2023D/EGamma1/NANOAOD/22Sep2023_v2-v1/40000/6ab2c1d9-5721-4472-841a-dda62aa5fc47.root': 'Events'},
            'metadata': {'short_name': 'SingleEle'}}},
        'SingleMuonD': {'Muon1v2D_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/data/Run2023D/Muon1/NANOAOD/22Sep2023_v2-v1/50000/9aa57c83-6e4d-4254-ba43-4fde90576279.root': 'Events'},
            'metadata': {'short_name': 'SingleMuon'}}},
    },
    '2024': {
        'SingleMuonC': {'Muon0Run2024Cv1_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/data/Run2024C/Muon0/NANOAOD/2024CDEReprocessing-v1/2550000/f5535e1c-a482-492d-9fab-47d46b75ee10.root': 'Events'},
            'metadata': {'short_name': 'SingleMuon'}}},
        'SingleMuonF': {'Muon1F_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/data/Run2024F/Muon1/NANOAOD/PromptReco-v1/000/383/767/00000/4cc4fcb9-75f1-46e6-bfa2-bba3167a3e47.root': 'Events'},
            'metadata': {'short_name': 'SingleMuon'}}},
        'EGammaI': {'EGamma0I_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/data/Run2024I/EGamma0/NANOAOD/PromptReco-v1/000/386/693/00000/bda6c53a-708a-4974-baaa-9a3cceba89bf.root': 'Events'},
            'metadata': {'short_name': 'SingleEle'}}},
        'DYJetsToLL': {'DYJetsToLL_50to120_HT40to70_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/RunIII2024Summer24NanoAODv15/DYto2L-4Jets_Bin-HT-40to70-MLL-50to120_TuneCP5_13p6TeV_madgraphMLM-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v2/2550000/74f3b333-8f05-4860-b99d-f2aa3e1d7054.root': 'Events'},
            'metadata': {'short_name': 'DYJetsToLL'}}},
        'DYJetsToLL_inclusive': {'DYto2Mu_inclusive_50_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/RunIII2024Summer24NanoAODv15/DYto2Mu-2Jets_Bin-MLL-50_TuneCP5_13p6TeV_amcatnloFXFX-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v6/100000/0d805200-1e12-4bca-9388-bf3dd3165cc5.root': 'Events'},
            'metadata': {'short_name': 'DYJetsToLL'}}},
        'TTto2L2Nu': {'TTto2L2Nu_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/RunIII2024Summer24NanoAODv15/TTto2L2Nu_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v3/2810000/79563aaa-146c-4299-a375-c8a9c95a5eed.root': 'Events'},
            'metadata': {'short_name': 'tt'}}}, 
        'TTtoLNu2Q': {'TTtoLNu2Q_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/RunIII2024Summer24NanoAODv15/TTtoLNu2Q_TuneCP5_13p6TeV_powheg-pythia8/NANOAODSIM/150X_mcRun3_2024_realistic_v2-v2/2520000/02e98655-c4b3-4a60-9cc4-8cc6be94e173.root': 'Events'},
            'metadata': {'short_name': 'tt'}}}, 
        'ZZ': {'ZZ_1': {
            'files': {'root://cms-xrd-global.cern.ch//store/mc/Run3Winter26NanoAODv15/ZZ_pomflux-H1FitB_TuneCP5_13p6TeV-pythia8/NANOAODSIM/LowPUAVE5_150X_mcRun3_realistic_EOY25_forTSGStudies_v3-v2/100000/2f237cbc-d489-4848-a4f7-8f53f857cf0b.root': 'Events'},
            'metadata': {'short_name': 'Diboson'}}},
    }
}

In [7]:
workflow = "btag_eff_ele"
year = "2022preEE"
dataset = "TTTo2L2Nu"

In [8]:
preprocessed_fileset, updated_fileset = preprocess(
    fileset[year][dataset],
    align_clusters = True,
    skip_bad_files = False,
    #step_size = 100_000,
)
futures_run = processor.Runner(
    executor=processor.FuturesExecutor(workers=4, compression=None, retries=1),
    schema=NanoAODSchema,
    #chunksize=100000,
    savemetrics=True,
)
out, metrics = futures_run(
    preprocessed_fileset, 
    treename="Events", 
    processor_instance=BaseProcessor(workflow=workflow, year=year, mode="virtual"),
)
out["metadata"]

Output()

{'sumw': np.float32(3.5448078e+06),
 'base': {'cutflow': {'initial': np.float32(3.5448078e+06),
   'goodvertex': np.float32(3.3598095e+06),
   'lumi': np.float32(3.3598095e+06),
   'trigger': np.float32(1.0558105e+06),
   'trigger_match': np.float32(1.0451047e+06),
   'metfilters': np.float32(1.0446181e+06),
   'hemcleaning': np.float32(1.0446181e+06),
   'met_50': np.float32(742100.56),
   'dy_stitching': np.float32(742100.56),
   'w_stitching': np.float32(742100.56),
   'tau_veto': np.float32(742100.56),
   'muon_veto': np.float32(598303.4),
   'exactly_one_electron': np.float32(283295.88)},
  'weighted_final_nevents': np.float64(254779.29470871916)}}

In [4]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
filename = list(list(fileset[year][dataset].values())[0]['files'].keys())[0]
events = NanoEventsFactory.from_root(
    filename,
    treepath="Events",
    entry_stop=1_000,
    metadata={"dataset": "WJetsToLNu_0to120_HT100to400_1"},
    schemaclass=NanoAODSchema,
    mode="virtual",
).events()

In [ ]:
from coffea.nanoevents import NanoEventsFactory, NanoAODSchema
filename = "root://cms-xrd-global.cern.ch//store/data/Run2024C/Muon0/NANOAOD/2024CDEReprocessing-v1/2550000/f5535e1c-a482-492d-9fab-47d46b75ee10.root"
events = NanoEventsFactory.from_root(
    filename,
    treepath="Events",
    entry_stop=1_000,
    metadata={"dataset": "Muon0"},
    schemaclass=NanoAODSchema,
    mode="virtual",
).events()
electrons = events.Electron
hasattr(electrons,'superclusterEta')

False